<a href="https://colab.research.google.com/github/shay2222/ml-systems-lab/blob/main/01-kv-cache.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Sat Oct 17 23:33:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")
print("Model loaded")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [4]:
import time

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Write a short story about a robot."}],
    tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

start = time.time()
output = model.generate(**inputs, max_new_tokens=200)
elapsed = time.time() - start

new_tokens = output.shape[1] - inputs["input_ids"].shape[1]
print(f"Generated {new_tokens} tokens in {elapsed:.2f} seconds")
print(f"Speed: {new_tokens / elapsed:.1f} tokens per second")

Generated 200 tokens in 8.81 seconds
Speed: 22.7 tokens per second


In [5]:
import time

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Write a short story about a robot."}],
    tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

speeds = []
for run in range(5):
    start = time.time()
    output = model.generate(**inputs, max_new_tokens=200)
    elapsed = time.time() - start
    new_tokens = output.shape[1] - inputs["input_ids"].shape[1]
    speed = new_tokens / elapsed
    speeds.append(speed)
    print(f"Run {run + 1}: {speed:.1f} tokens per second")

print(f"Average: {sum(speeds) / len(speeds):.1f} tokens per second")

Run 1: 20.7 tokens per second
Run 2: 28.7 tokens per second
Run 3: 28.5 tokens per second
Run 4: 28.7 tokens per second
Run 5: 28.4 tokens per second
Average: 27.0 tokens per second


Experiment 01: Does the KV cache change speed?

A KV cache stores attention results from earlier tokens so the model doesn’t recompute them for every new token. Without it, generation gets slower as the text gets longer. Hugging Face turns the cache on by default, so you’ll measure it both ways.

In [6]:
import time

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

def measure(use_cache, runs=5, tokens=200):
    speeds = []
    for _ in range(runs):
        start = time.time()
        out = model.generate(
            **inputs,
            max_new_tokens=tokens,
            use_cache=use_cache,
            do_sample=False,  # same output every run, so the test is fair
        )
        elapsed = time.time() - start
        new = out.shape[1] - inputs["input_ids"].shape[1]
        speeds.append(new / elapsed)
    return sum(speeds) / len(speeds)

# Warm-up run, not counted
model.generate(**inputs, max_new_tokens=20, use_cache=True, do_sample=False)

with_cache = measure(use_cache=True)
without_cache = measure(use_cache=False)

print(f"With KV cache:    {with_cache:.1f} tokens per second")
print(f"Without KV cache: {without_cache:.1f} tokens per second")
print(f"Speedup: {with_cache / without_cache:.1f}x")

With KV cache:    27.3 tokens per second
Without KV cache: 26.8 tokens per second
Speedup: 1.0x


Show no real difference at 200 tokens: 27.3 with the cache and 26.8 without, a 1.0x speedup. That’s a valid result, not a mistake. The KV cache saves work that grows with the length of the text, so at 200 tokens on a small model, the saving is too small to see above the normal run-to-run variation.

In [11]:
import time

def measure(use_cache, tokens, runs=3):
    speeds = []
    for _ in range(runs):
        start = time.time()
        out = model.generate(
            **inputs,
            max_new_tokens=tokens,
            min_new_tokens=tokens,
            use_cache=use_cache,
            do_sample=False,
        )
        elapsed = time.time() - start
        new = out.shape[1] - inputs["input_ids"].shape[1]
        speeds.append(new / elapsed)
    return sum(speeds) / len(speeds)

print("Tokens | With cache | Without cache | Speedup")
for tokens in [200, 500, 700]:
    w = measure(True, tokens)
    wo = measure(False, tokens)
    print(f"{tokens:6} | {w:10.1f} | {wo:13.1f} | {w/wo:.2f}x")



Tokens | With cache | Without cache | Speedup
   200 |       18.0 |          27.1 | 0.66x
   500 |       26.7 |          24.4 | 1.09x
   700 |       29.5 |          19.5 | 1.51x


At 700 tokens, the KV cache is 1.51x faster. At 200 tokens it’s about the same, and at 500 it’s slightly faster. The pattern holds: the cache helps more as the output gets longer. The 200-token result changed between runs (0.97x, 0.67x, 0.66x), so that row is mostly noise.